### Customer churn prediction

In [1]:

import pandas as pd

# Load the IBM Telco Customer Churn dataset
url = (
    "https://raw.githubusercontent.com/"
    "IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)

df = pd.read_csv(url)

# Save a local copy
df.to_csv("Telco-Customer-Churn.csv", index=False)

# Inspect the dataset
print("Dataset shape:", df.shape)

print("\nFirst 5 rows:")
print(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nChurn class counts:")
print(df["Churn"].value_counts())

print("\nMissing values per column:")
print(df.isnull().sum())


Dataset shape: (7043, 21)

First 5 rows:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport Streami

In [2]:

# 1. Remove unnecessary spaces from column names
df.columns = df.columns.str.strip()

# 2. Remove duplicate records
print("Duplicates before cleaning:", df.duplicated().sum())
df = df.drop_duplicates()

# 3. Convert TotalCharges into numeric values
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# 4. Check missing values
print("\nMissing values after conversion:")
print(df.isnull().sum())

# 5. Check rows where TotalCharges is missing
print("\nRows with missing TotalCharges:")
print(df[df["TotalCharges"].isnull()][
    ["tenure", "MonthlyCharges", "TotalCharges"]
])

# 6. Fill missing TotalCharges with 0
df["TotalCharges"] = df["TotalCharges"].fillna(0)

print("\nMissing values after cleaning:")
print(df.isnull().sum().sum())

print("\nCleaned dataset shape:", df.shape)


Duplicates before cleaning: 0

Missing values after conversion:
customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

Rows with missing TotalCharges:
      tenure  MonthlyCharges  TotalCharges
488        0           52.55           NaN
753        0           20.25           NaN
936        0           80.85           NaN
1082       0           25.75           NaN
1340       0           56.05           NaN
3331       0           19.85           NaN
3826       0           25.35           NaN
4380       0           20.00           NaN
5218       0        

In [4]:
print("Target values:")
print(df["Churn"].value_counts())

print("\nCategorical columns:")
print(df.select_dtypes(include="object").columns.tolist())

print("\nNumerical columns:")
print(df.select_dtypes(include="number").columns.tolist())

Target values:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Categorical columns:
['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']

Numerical columns:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


C:\Users\Varshini D\AppData\Local\Temp\ipykernel_20780\3624223038.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.select_dtypes(include="object").columns.tolist())


In [5]:

# STEP 3: Feature Engineering and Encoding

# 1. Remove customerID because it is only an identifier
df = df.drop(columns=["customerID"])

# 2. Convert target column into numerical values
# No = 0, Yes = 1
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

# 3. Separate features (X) and target (y)
X = df.drop(columns=["Churn"])
y = df["Churn"]

# 4. Convert categorical features into numerical columns
X = pd.get_dummies(
    X,
    drop_first=True,
    dtype=int
)

# 5. Check the results
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nFirst 5 encoded rows:")
print(X.head())

print("\nTarget values:")
print(y.value_counts())

print("\nAny missing values in features:", X.isnull().sum().sum())
print("Any missing values in target:", y.isnull().sum())


Feature shape: (7043, 30)
Target shape: (7043,)

First 5 encoded rows:
   SeniorCitizen  tenure  MonthlyCharges  TotalCharges  gender_Male  \
0              0       1           29.85         29.85            0   
1              0      34           56.95       1889.50            1   
2              0       2           53.85        108.15            1   
3              0      45           42.30       1840.75            1   
4              0       2           70.70        151.65            0   

   Partner_Yes  Dependents_Yes  PhoneService_Yes  \
0            1               0                 0   
1            0               0                 1   
2            0               0                 1   
3            0               0                 0   
4            0               0                 1   

   MultipleLines_No phone service  MultipleLines_Yes  ...  \
0                               1                  0  ...   
1                               0                  0  ...   
2     

In [6]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# STEP 4: Split the dataset

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training targets:", y_train.shape)
print("Testing targets:", y_test.shape)

# STEP 5: Feature Scaling

scaler = StandardScaler()

# Learn scaling parameters from training data only
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same parameters to test data
X_test_scaled = scaler.transform(X_test)

print("\nScaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)


Training features: (5634, 30)
Testing features: (1409, 30)
Training targets: (5634,)
Testing targets: (1409,)

Scaled training shape: (5634, 30)
Scaled testing shape: (1409, 30)


In [7]:

import torch

# STEP 5: Convert NumPy arrays into PyTorch tensors

X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)

y_test_tensor = torch.tensor(
    y_test.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)

# Check tensor shapes and data types
print("X_train:", X_train_tensor.shape)
print("X_test:", X_test_tensor.shape)

print("y_train:", y_train_tensor.shape)
print("y_test:", y_test_tensor.shape)

print("\nFeature tensor dtype:", X_train_tensor.dtype)
print("Target tensor dtype:", y_train_tensor.dtype)


X_train: torch.Size([5634, 30])
X_test: torch.Size([1409, 30])
y_train: torch.Size([5634, 1])
y_test: torch.Size([1409, 1])

Feature tensor dtype: torch.float32
Target tensor dtype: torch.float32


In [8]:

import torch.nn as nn

# STEP 6: Build the ANN model

input_size = X_train_tensor.shape[1]

class ChurnModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)


# Create the model
model = ChurnModel()

print(model)
print("\nNumber of input features:", input_size)


ChurnModel(
  (network): Sequential(
    (0): Linear(in_features=30, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): ReLU()
    (5): Linear(in_features=16, out_features=1, bias=True)
  )
)

Number of input features: 30


In [9]:

# STEP 7: Loss Function and Optimizer

# Binary classification loss
criterion = nn.BCEWithLogitsLoss()

# Optimizer updates the model's weights
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)
print("Learning rate:", 0.001)


Loss function: BCEWithLogitsLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
Learning rate: 0.001


In [10]:

# STEP 8: Train the Neural Network

epochs = 100

for epoch in range(epochs):

    # Training mode: enables dropout
    model.train()

    # Clear gradients from the previous iteration
    optimizer.zero_grad()

    # 1. Forward pass
    outputs = model(X_train_tensor)

    # 2. Calculate loss
    loss = criterion(outputs, y_train_tensor)

    # 3. Backpropagation
    loss.backward()

    # 4. Update weights
    optimizer.step()

    # Display progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch [{epoch + 1}/{epochs}], "
            f"Loss: {loss.item():.4f}"
        )

print("\nTraining completed!")


Epoch [10/100], Loss: 0.6179
Epoch [20/100], Loss: 0.5779
Epoch [30/100], Loss: 0.5378
Epoch [40/100], Loss: 0.5011
Epoch [50/100], Loss: 0.4754
Epoch [60/100], Loss: 0.4594
Epoch [70/100], Loss: 0.4509
Epoch [80/100], Loss: 0.4428
Epoch [90/100], Loss: 0.4355
Epoch [100/100], Loss: 0.4343

Training completed!


In [13]:

# STEP 9: Evaluate the model

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Switch to evaluation mode
model.eval()

# Disable gradient calculations during testing
with torch.no_grad():
    test_logits = model(X_test_tensor)

    # Convert logits into probabilities
    test_probabilities = torch.sigmoid(test_logits)

    # Convert probabilities into class labels
    test_predictions = (test_probabilities >= 0.5).int()

# Convert tensors into NumPy arrays
y_true = y_test_tensor.numpy().ravel()
y_pred = test_predictions.numpy().ravel()
y_prob = test_probabilities.numpy().ravel()

# Calculate evaluation metrics
accuracy = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

print("\nTest Accuracy:", round(accuracy, 4))
print("ROC-AUC Score:", round(auc, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["No Churn", "Churn"],
        zero_division=0
    )
)



Test Accuracy: 0.7892
ROC-AUC Score: 0.8248

Confusion Matrix:
[[906 129]
 [168 206]]

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.84      0.88      0.86      1035
       Churn       0.61      0.55      0.58       374

    accuracy                           0.79      1409
   macro avg       0.73      0.71      0.72      1409
weighted avg       0.78      0.79      0.79      1409



In [15]:
import json

with open("feature_columns.json", "w") as file:
    json.dump(X.columns.tolist(), file)

In [16]:

# STEP 10: Save the trained model and scaler

import pickle

# Save the trained model weights
torch.save(
    model.state_dict(),
    "churn_model.pth"
)

# Save the fitted scaler
with open("churn_scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

print("Model saved: churn_model.pth")
print("Scaler saved: churn_scaler.pkl")


Model saved: churn_model.pth
Scaler saved: churn_scaler.pkl
